In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import warnings
warnings.filterwarnings("ignore")



#### CHAIN WITH Parallel Chains

In [2]:
# TASK -1 [Prompt]

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer"),
    ("human", "Please summarize the movie in brief : {input}")
])

In [3]:
# TASK - 2 [LLM]

llm = ChatOllama(
    model="mistral:latest",
    temperature=0.7
)

In [4]:
# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

In [20]:
# TASK - 4 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text:str)-> dict:

    return {"text" : text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

#### Parallel Chain 1  == > Linked in

In [ ]:
# TASK - 1 [Prompt]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")])

In [7]:
# TASK - 2 [LLM]

llm = ChatOllama(
    model="mistral:latest",
    temperature=0.7
)



In [8]:
# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

In [11]:
# Chain

chain_linkedin = linkedin_prompt | llm | str_parser

#### Parallel Chain 2

In [10]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

In [13]:
def insta_chain(text:dict):

    text=text['text']

    # TASK - 1 [Prompt]

    linkedin_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a LinkedIn post generator"),
        ("human", "Create a post for the following text for LinkedIn: {text}")])

    # TASK - 2 [LLM]

    llm = ChatOllama(
        model="mistral:latest",
        temperature=0.7
    )

    # TASK - 3 [Str Parser]

    str_parser = StrOutputParser()

    chain_insta = linkedin_prompt | llm | str_parser

    result = chain_insta.invoke(text)

    return result

insta_runable = RunnableLambda(insta_chain)

    

#### **Final Orchestration**"

In [23]:
from langchain_core.runnables import branch
final_chain=(
    prompt_template | 
    llm | 
    str_parser |
    dictionary_maker_runnable |
    RunnableParallel(branch={'linked in':chain_linkedin, 'instagram':insta_runable})
)

In [25]:
final_chain.invoke("KGF")

{'branch': {'linked in': ' Title: 🌟Unleashing the Power of Resilience and Determination in KGF 🌟\n\n🎥️ Dive into the action-packed world of "KGF" - a captivating film that masterfully portrays the transformative journey of Rocky (Yash), an orphan from humble beginnings, to becoming the undisputed kingpin of Mumbai\'s underworld.\n\nOriginally hailing from Kolar Gold Fields in Karnataka, India, Rocky is nurtured and mentored by the wise Anand Ingalagi (Srinivasa Murthy). Following Anand\'s demise, Rocky embarks on a mission to claim the throne of the gold mines of Kolar.\n\nRocky\'s path is fraught with challenges from ruthless local politicians, relentless rival gangsters, and formidable drug lord Garuda (Anant Nag). Yet, Rocky\'s unyielding determination and tactical prowess propel him to seize control of the mines, earning him the revered title \'Gandhabba\'.\n\nAs we eagerly await the sequel, be prepared for Rocky to face a more formidable adversary in Reena Desai (Srinidhi Shetty).

#### **Chain as a Runnable**

In [28]:
# TASK - 1 [Beautify Function]

def beautify(final_response:dict)-> dict:

    linkedin_response = final_response['branch']['linked in']
    instagram_response = final_response['branch']['instagram']

    return {"linkedin": linkedin_response, "instagram": instagram_response}

beautify_runnable = RunnableLambda(beautify)


# TASK - 2 [Final Chain]

# final_chain 


# Beautified Chain
beautified_chain = final_chain | beautify_runnable

beautified_chain.invoke("Pushpa")

{'linkedin': ' Title: 🌲💥 Embracing the Roar of "Pushpa" 💥🌲\n\nExcited to share my thoughts on the riveting action-drama, "Pushpa"! Set against the enigmatic Seshachalam forests of South India, this film offers an immersive journey into the world of red sandalwood smuggling mafia.\n\nThe narrative revolves around Pushpa Raj (Allu Arjun), a humble laborer with dreams as vast as the forest itself. His life takes a dramatic turn when he stumbles upon a treasure trove of valuable red sandalwood, propelling him into the upper echelons of the mafia.\n\nAs Pushpa\'s ambitions ignite conflicts and power struggles, the film delves deep into themes of power, corruption, love, and survival. It paints a gritty picture of a world where loyalty is often questioned, and danger lurks around every corner.\n\nI find myself captivated by the raw energy, compelling performances, and the cinematic brilliance that makes "Pushpa" a must-watch! 🎥✨ #Pushpa #RedSandalwoodMafia #ActionDrama #AlluArjun #SouthIndia